# 04 - Zillow Data Pipeline

Loads Zillow ZHVI county-level data, assigns Karl & Koss climate regions, and computes
region-minus-affected baselines for each county-month storm observation.

**Inputs:**
- Zillow ZHVI county CSV (fetched directly)
- `../data/processed/storm_events.pkl` — to identify affected counties per month

**Output:** `../data/processed/zillow_panel.pkl` — one row per county-month storm observation
with ZHVI indexed to 100 at storm month and regional baseline ZHVI for comparison

**Baseline definition:** Equal-weighted mean ZHVI of all unaffected counties adjacent to the target county


In [1]:
import pandas as pd
import numpy as np
from pathlib import Path
import urllib

Path('../data/raw').mkdir(parents=True, exist_ok=True)
Path('../data/processed').mkdir(parents=True, exist_ok=True)

## ZillowDataParser
Unchanged from `NOAA_Storm_and_Zillow_Data_Pipeline.ipynb`

In [2]:
class ZillowDataParser():
    DTYPES = {
        'RegionID': 'Int64',
        'SizeRank': 'Int64',
        'RegionName': 'object',
        'RegionType': 'object',
        'StateName': 'object',
        'State': 'object',
        'Metro': 'object',
        'StateCodeFIPS': 'Int64',
        'MunicipalCodeFIPS': 'Int64'
    }
    
    META_DATA_COLS = ['RegionID', 'SizeRank', 'RegionName', 'RegionType',
                      'StateName', 'State', 'Metro', 'StateCodeFIPS', 'MunicipalCodeFIPS',
                      'latitude', 'longitude']
    
    def __init__(self, regional_url, national_url):
        self.download_data(regional_url, national_url)
        
    def download_data(self, regional_url, national_url):
        regional_data_raw = pd.read_csv(regional_url)
        national_data_raw = pd.read_csv(national_url)
        national = national_data_raw.loc[national_data_raw["RegionName"] == "United States"]
        self.df = pd.concat([regional_data_raw, national], sort=False, ignore_index=True)
        self.df = self.df.astype(self.DTYPES)
        all_columns = self.df.columns.tolist() + [col for col in self.META_DATA_COLS if col not in self.df.columns]
        self.df = self.df.reindex(columns=all_columns)

    def date_cols(self):
        dates = [col for col in self.df.columns if col not in self.META_DATA_COLS]
        dates = sorted(dates, key=pd.to_datetime)
        return list(dates)

    def get_monthly_panel(self):
        date_cols = self.date_cols()
        long_df = self.df.melt(
            id_vars=self.META_DATA_COLS,
            value_vars=date_cols,
            var_name='date',
            value_name='zhvi'
        )
        long_df['date'] = pd.to_datetime(long_df['date'])
        return long_df

In [3]:
zillow_county_url        = "https://files.zillowstatic.com/research/public_csvs/zhvi/County_zhvi_uc_sfrcondo_tier_0.33_0.67_sm_sa_month.csv"
zillow_county_top_url    = "https://files.zillowstatic.com/research/public_csvs/zhvi/County_zhvi_uc_sfrcondo_tier_0.67_1.0_sm_sa_month.csv"
zillow_county_bottom_url = "https://files.zillowstatic.com/research/public_csvs/zhvi/County_zhvi_uc_sfrcondo_tier_0.0_0.33_sm_sa_month.csv"
zillow_msa_url           = "https://files.zillowstatic.com/research/public_csvs/zhvi/Metro_zhvi_uc_sfrcondo_tier_0.33_0.67_sm_sa_month.csv"## Load Zillow Data

county_mid_path    = Path('../data/raw/zillow_county_mid_zhvi.csv')
county_top_path    = Path('../data/raw/zillow_county_top_zhvi.csv')
county_bottom_path = Path('../data/raw/zillow_county_bottom_zhvi.csv')
msa_path           = Path('../data/raw/zillow_msa_zhvi.csv')

if not county_mid_path.exists():
    urllib.request.urlretrieve(zillow_county_url, county_mid_path)
    print('Downloaded mid tier')
else:
    print('Using cached mid tier')

if not county_top_path.exists():
    urllib.request.urlretrieve(zillow_county_top_url, county_top_path)
    print('Downloaded top tier')
else:
    print('Using cached top tier')

if not county_bottom_path.exists():
    urllib.request.urlretrieve(zillow_county_bottom_url, county_bottom_path)
    print('Downloaded bottom tier')
else:
    print('Using cached bottom tier')

if not msa_path.exists():
    urllib.request.urlretrieve(zillow_msa_url, msa_path)
    print('Downloaded MSA')
else:
    print('Using cached MSA')

zillow_mid    = ZillowDataParser(str(county_mid_path), str(msa_path))
zillow_top    = ZillowDataParser(str(county_top_path), str(msa_path))
zillow_bottom = ZillowDataParser(str(county_bottom_path), str(msa_path))

print(f'Mid:    {len(zillow_mid.df):,} rows, {len(zillow_mid.date_cols())} months')
print(f'Top:    {len(zillow_top.df):,} rows, {len(zillow_top.date_cols())} months')
print(f'Bottom: {len(zillow_bottom.df):,} rows, {len(zillow_bottom.date_cols())} months')

Using cached mid tier
Using cached top tier
Using cached bottom tier
Using cached MSA
Mid:    3,074 rows, 315 months
Top:    3,077 rows, 315 months
Bottom: 3,002 rows, 315 months


## Build Long Panel and Assign Climate Regions

In [4]:
def build_panel(zillow):
    panel = zillow.get_monthly_panel()

    # Keep only county-level rows (drop national, MSA etc)
    panel = panel[panel['RegionType'] == 'county'].copy()

    # Build stcofips
    panel['state_fips']  = panel['StateCodeFIPS'].astype(str).str.zfill(2)
    panel['county_fips'] = panel['MunicipalCodeFIPS'].astype(str).str.zfill(3)
    panel['stcofips']    = panel['state_fips'] + panel['county_fips']

    # Extract year and month
    panel['year']  = panel['date'].dt.year
    panel['month'] = panel['date'].dt.month


    # Drop PR, AK, HI
    EXCLUDE_STATES = {'02', '15', '72'}
    panel = panel[~panel['state_fips'].isin(EXCLUDE_STATES)].copy()

    zhvi_idx = panel.set_index(['stcofips', 'year', 'month'])['zhvi']

    # Filter to storm event years only
    panel = panel[panel['year'].between(2020, 2025)].copy()

    # Drop rows with no ZHVI or no region assignment
    n_before = len(panel)
    panel = panel.dropna(subset=['zhvi'])
    print(f'Dropped {n_before - len(panel):,} rows with missing ZHVI or unassigned region')
    print(f'Panel shape: {panel.shape}')

    return panel,zhvi_idx

panel_mid,    zhvi_idx_mid    = build_panel(zillow_mid)
panel_top,    zhvi_idx_top    = build_panel(zillow_top)
panel_bottom, zhvi_idx_bottom = build_panel(zillow_bottom)

Dropped 1,657 rows with missing ZHVI or unassigned region
Panel shape: (218303, 18)
Dropped 1,154 rows with missing ZHVI or unassigned region
Panel shape: (219022, 18)
Dropped 2,838 rows with missing ZHVI or unassigned region
Panel shape: (212010, 18)


In [5]:
zhvi_idx_mid.to_pickle('../data/processed/zhvi_idx_mid.pkl')
zhvi_idx_top.to_pickle('../data/processed/zhvi_idx_top.pkl')
zhvi_idx_bottom.to_pickle('../data/processed/zhvi_idx_bottom.pkl')
print('Saved zhvi_idx pickles')

Saved zhvi_idx pickles


## Load Storm Events and Compute Neighbor-Unaffected Baseline

In [6]:
storms = pd.read_pickle('../data/processed/storm_events.pkl')
# Set of affected county-month pairs
affected = set(zip(storms['stcofips'], storms['year'], storms['month']))
print(f'Affected county-month observations: {len(affected):,}')

def affected_panel(panel,storms):

    # Flag affected counties in the full panel
    panel['affected'] = panel.apply(
        lambda r: (r['stcofips'], r['year'], r['month']) in affected, axis=1
    )

    print(f'Affected rows in Zillow panel: {panel["affected"].sum():,}')
    print(f'Unaffected rows (baseline pool): {(~panel["affected"]).sum():,}')
    return panel

panel_mid    = affected_panel(panel_mid,    storms)
panel_top    = affected_panel(panel_top,    storms)
panel_bottom = affected_panel(panel_bottom, storms)

Affected county-month observations: 48,978
Affected rows in Zillow panel: 47,261
Unaffected rows (baseline pool): 171,042
Affected rows in Zillow panel: 47,362
Unaffected rows (baseline pool): 171,660
Affected rows in Zillow panel: 46,101
Unaffected rows (baseline pool): 165,909


In [7]:
# Adjacent Counties Baseline
# Compute baseline from local unaffected counties
adj_df = pd.read_csv('../data/raw/county_adjacency.csv')
adj_df['County GEOID'] = adj_df['County GEOID'].astype(int).astype(str).str.zfill(5)
adj_df['Neighbor GEOID'] = adj_df['Neighbor GEOID'].astype(int).astype(str).str.zfill(5)

In [8]:
NEIGHBOR_LEVEL = 1
def create_neighbors(zhvi_idx):
    zhvi_fips = set(zhvi_idx.index.get_level_values('stcofips'))

    if NEIGHBOR_LEVEL == 1:
        # Create a dictionary where key = County, value = list of Neighbors
        # We exclude the county itself if it appears in the neighbors list
        neighbors = adj_df[adj_df['Neighbor GEOID'] != adj_df['County GEOID']].groupby('County GEOID')['Neighbor GEOID'].apply(list).to_dict()
        

    elif NEIGHBOR_LEVEL == 2:
        # Build layer 1
        layer1 = adj_df[adj_df['Neighbor GEOID'] != adj_df['County GEOID']][["County GEOID", "Neighbor GEOID"]]
        layer1_dict = layer1.groupby('County GEOID')['Neighbor GEOID'].apply(set).to_dict()
        
        # Build layer 2 via self-join
        neighbors = layer1.merge(
            layer1,
            left_on  = 'Neighbor GEOID',
            right_on = 'County GEOID'
        )
        neighbors = neighbors.drop(["Neighbor GEOID_x", "County GEOID_y"], axis=1)
        neighbors = neighbors.rename(columns={"County GEOID_x": "target", "Neighbor GEOID_y": "layer2"})
        
        # Exclude target itself and layer 1 neighbors
        neighbors = neighbors[
            (neighbors["target"] != neighbors["layer2"]) &
            (~neighbors.apply(lambda r: r["layer2"] in layer1_dict.get(r["target"], set()), axis=1))
        ]
        neighbors = neighbors.groupby("target")["layer2"].apply(list).to_dict()
    else:
        raise ValueError("Neighbor level cannot be greater than 2")

    return {
        target: [n for n in neighbor_list if n in zhvi_fips]
        for target, neighbor_list in neighbors.items()
    }
neighbors_mid    = create_neighbors(zhvi_idx_mid)
neighbors_top    = create_neighbors(zhvi_idx_top)
neighbors_bottom = create_neighbors(zhvi_idx_bottom)

In [9]:
PRE_EVENT_MONTHS  = 3
POST_EVENT_MONTHS = 9
MIN_NEIGHBORS     = 1

def get_window_months(storm_year, storm_month, pre, post):
    """Returns list of (t, year, month) for t in [-pre, ..., +post]"""
    storm_date = pd.Timestamp(year=storm_year, month=storm_month, day=1)
    return [
        (t, (storm_date + pd.DateOffset(months=t)).year,
            (storm_date + pd.DateOffset(months=t)).month)
        for t in range(-pre, post + 1)
    ]
def create_baseline_lookup(neighbors, zhvi_idx):
    rows    = []
    flagged = 0

    for (target_fips, storm_year, storm_month) in affected:
        neighbor_list = neighbors.get(target_fips, [])
        if not neighbor_list:
            continue

        window = get_window_months(storm_year, storm_month, PRE_EVENT_MONTHS, POST_EVENT_MONTHS)

        # Step 1: Keep neighbors unaffected across entire window
        clean_neighbors = [
            n for n in neighbor_list
            if all((n, yr, mo) not in affected for (_, yr, mo) in window)
        ]

        # Step 2: Keep neighbors with complete ZHVI coverage across entire window
        # including T=0 — required for consistent indexing
        complete_neighbors = []
        for n in clean_neighbors:
            zhvi_t0 = zhvi_idx.get((n, storm_year, storm_month))
            if zhvi_t0 is None or pd.isna(zhvi_t0) or zhvi_t0 == 0:
                continue
            if all(
                zhvi_idx.get((n, yr, mo)) is not None and
                not pd.isna(zhvi_idx.get((n, yr, mo)))
                for (_, yr, mo) in window
            ):
                complete_neighbors.append(n)

        n_clean = len(complete_neighbors)
        if n_clean < MIN_NEIGHBORS:
            flagged += 1
        if n_clean == 0:
            continue

        # Step 3: For each offset t, average individually indexed neighbor values
        for (t, yr, mo) in window:
            indexed_vals = []
            for n in complete_neighbors:
                zhvi_t0 = zhvi_idx.get((n, storm_year, storm_month))
                zhvi_t  = zhvi_idx.get((n, yr, mo))
                indexed_vals.append((zhvi_t / zhvi_t0) * 100)

            rows.append({
                'target_fips':       target_fips,
                'storm_year':        storm_year,
                'storm_month':       storm_month,
                't':                 t,
                'year':              yr,
                'month':             mo,
                'baseline_zhvi':     np.mean(indexed_vals),
                'n_clean_neighbors': n_clean,
                'neighbor_fips':     ','.join(complete_neighbors)
            })

    baseline_lookup = pd.DataFrame(rows)
    print(f'Total baseline rows:                    {len(baseline_lookup):,}')
    print(f'Events with <{MIN_NEIGHBORS} clean neighbors (flagged): {flagged:,}')
    print(f'Storm events covered:                   {baseline_lookup.groupby(["target_fips","storm_year","storm_month"]).ngroups:,}')
    return baseline_lookup

baseline_lookup_mid    = create_baseline_lookup(neighbors_mid, zhvi_idx_mid)
baseline_lookup_top    = create_baseline_lookup(neighbors_top, zhvi_idx_top)
baseline_lookup_bottom = create_baseline_lookup(neighbors_bottom, zhvi_idx_bottom)

Total baseline rows:                    119,106
Events with <1 clean neighbors (flagged): 38,874
Storm events covered:                   9,162
Total baseline rows:                    119,548
Events with <1 clean neighbors (flagged): 38,840
Storm events covered:                   9,196
Total baseline rows:                    114,010
Events with <1 clean neighbors (flagged): 39,239
Storm events covered:                   8,770


## Validate

In [10]:
# Validate
for name, baseline_lookup in [
    ('mid',    baseline_lookup_mid),
    ('top',    baseline_lookup_top),
    ('bottom', baseline_lookup_bottom)
]:
    assert baseline_lookup['target_fips'].str.len().eq(5).all(), f'{name}: FIPS not all 5 digits'
    assert baseline_lookup['t'].between(-PRE_EVENT_MONTHS, POST_EVENT_MONTHS).all(), f'{name}: Offset t out of window range'
    assert baseline_lookup.duplicated(['target_fips', 'storm_year', 'storm_month', 't']).sum() == 0, f'{name}: Duplicate event-offset rows'
    missing = baseline_lookup['baseline_zhvi'].isnull().sum()
    if missing > 0:
        print(f'Warning ({name}): {missing:,} rows missing baseline_zhvi')
    else:
        print(f'{name}: All baseline ZHVI present')


mid: All baseline ZHVI present
top: All baseline ZHVI present
bottom: All baseline ZHVI present


## Export

In [11]:
def export_baseline(baseline_lookup, suff):
    baseline_lookup.to_pickle(f"../data/processed/baseline_lookup_{suff}.pkl")
    print(f"Saved baseline_lookup_{suff}.pkl")
    print(f'Shape: {baseline_lookup.shape}')
    print(f'Storm events covered: {baseline_lookup.groupby(["target_fips","storm_year","storm_month"]).ngroups:,}')

    print('All assertions passed')
    print(f'\nSample:')
    baseline_lookup[baseline_lookup['n_clean_neighbors'] >= MIN_NEIGHBORS].head(13)
    print(f'Total storm events in affected set: {len(affected):,}')
    print(f'Events in baseline_lookup: {baseline_lookup.groupby(["target_fips","storm_year","storm_month"]).ngroups:,}')
    print(f'Events with any null baseline: {baseline_lookup[baseline_lookup["baseline_zhvi"].isnull()].groupby(["target_fips","storm_year","storm_month"]).ngroups:,}')
    print(f'Events with complete windows: {baseline_lookup.groupby(["target_fips","storm_year","storm_month"]).filter(lambda x: x["baseline_zhvi"].notnull().all()).groupby(["target_fips","storm_year","storm_month"]).ngroups:,}')
export_baseline(baseline_lookup_mid,    'mid')
export_baseline(baseline_lookup_top,    'top')
export_baseline(baseline_lookup_bottom, 'bottom')

Saved baseline_lookup_mid.pkl
Shape: (119106, 9)
Storm events covered: 9,162
All assertions passed

Sample:
Total storm events in affected set: 48,978
Events in baseline_lookup: 9,162
Events with any null baseline: 0
Events with complete windows: 9,162
Saved baseline_lookup_top.pkl
Shape: (119548, 9)
Storm events covered: 9,196
All assertions passed

Sample:
Total storm events in affected set: 48,978
Events in baseline_lookup: 9,196
Events with any null baseline: 0
Events with complete windows: 9,196
Saved baseline_lookup_bottom.pkl
Shape: (114010, 9)
Storm events covered: 8,770
All assertions passed

Sample:
Total storm events in affected set: 48,978
Events in baseline_lookup: 8,770
Events with any null baseline: 0
Events with complete windows: 8,770
